In [1]:
!pip -q install pyspark

In [2]:
from google.colab import files
uploaded = files.upload()

Saving carrier_master.csv to carrier_master.csv
Saving route_master.csv to route_master.csv
Saving shipments_75000.csv to shipments_75000.csv
Saving shipments_extra_12000.csv to shipments_extra_12000.csv
Saving warehouse_master.csv to warehouse_master.csv
Saving warehouse_sensor_config.json to warehouse_sensor_config.json


In [3]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import *
from pyspark import StorageLevel
import os, time
spark = (SparkSession.builder
         .appName("LogisticsDeliveryPracticeSet02")
         .getOrCreate())
required_files = [
    "shipments_75000.csv",
    "shipments_extra_12000.csv",
    "warehouse_master.csv",
    "carrier_master.csv",
    "route_master.csv",
    "warehouse_sensor_config.json"
]
missing_files = [f for f in required_files if not os.path.exists(f)]
if missing_files:
    raise FileNotFoundError(f"Upload these files first: {missing_files}")
shipments_df = spark.read.option("header", True).option("inferSchema", True).csv("shipments_75000.csv")
extra_df = spark.read.option("header", True).option("inferSchema", True).csv("shipments_extra_12000.csv")
warehouse_df = spark.read.option("header", True).option("inferSchema", True).csv("warehouse_master.csv")
carrier_df = spark.read.option("header", True).option("inferSchema", True).csv("carrier_master.csv")
route_df = spark.read.option("header", True).option("inferSchema", True).csv("route_master.csv")
print("Main shipments:", shipments_df.count())
print("Extra shipments:", extra_df.count())
print("Warehouses:", warehouse_df.count())
print("Carriers:", carrier_df.count())
print("Routes:", route_df.count())

Main shipments: 75000
Extra shipments: 12000
Warehouses: 8
Carriers: 7
Routes: 12


In [4]:
shipments_df.printSchema()
print("Total records:", shipments_df.count())
shipments_df.show(15, truncate=False)

root
 |-- shipment_id: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)

Total records: 75000
+-----------+------------+-------------+--------------

In [5]:
shipments_df.select(
    "shipment_id", "warehouse_id", "service_level",
    "shipment_status", "shipping_fee_inr"
).show(truncate=False)

+-----------+------------+-------------+---------------+----------------+
|shipment_id|warehouse_id|service_level|shipment_status|shipping_fee_inr|
+-----------+------------+-------------+---------------+----------------+
|SHP0000001 |W08         |Express      |Delivered      |554.34          |
|SHP0000002 |W08         |Standard     |Delivered      |414.56          |
|SHP0000003 |W01         |Express      |Delivered      |724.64          |
|SHP0000004 |W03         |Standard     |Delivered      |287.86          |
|SHP0000005 |W03         |Economy      |Delivered      |251.68          |
|SHP0000006 |W01         |Express      |Delivered      |477.9           |
|SHP0000007 |W02         |Standard     |In Transit     |298.68          |
|SHP0000008 |W03         |Economy      |In Transit     |284.52          |
|SHP0000009 |W02         |Standard     |Delivered      |861.74          |
|SHP0000010 |W02         |Economy      |Delivered      |309.76          |
|SHP0000011 |W04         |Standard    

In [6]:
shipments_df.filter(
    (F.col("service_level") == "Express") &
    (F.col("package_weight_kg") > 5)
).show(truncate=False)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|SHP0000003 |2026-01-31  |2026-02-01   |2026-02-03            |2

In [7]:
shipments_df.orderBy(F.col("shipping_fee_inr").desc()).show(20, truncate=False)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|SHP0044904 |2026-08-11  |2026-08-13   |2026-08-16            |N

In [8]:
shipments_df.groupBy("shipment_status").count().orderBy(F.col("count").desc()).show()

+---------------+-----+
|shipment_status|count|
+---------------+-----+
|      Delivered|52434|
|     In Transit|10472|
|       Returned| 6130|
|      Cancelled| 3723|
|           Lost| 2241|
+---------------+-----+



In [9]:
service_summary = (
    shipments_df.groupBy("service_level")
    .agg(
        F.sum("shipping_fee_inr").alias("total_shipping_fees_inr"),
        F.sum("shipping_cost_inr").alias("total_shipping_costs_inr"),
        F.count("shipping_cost_inr").alias("shipments_with_known_cost")
    )
    .withColumn(
        "profit_inr",
        F.col("total_shipping_fees_inr") - F.col("total_shipping_costs_inr")
    )
    .withColumn(
        "profit_margin_pct",
        F.when(F.col("total_shipping_fees_inr") != 0,
               F.col("profit_inr") / F.col("total_shipping_fees_inr") * 100)
    )
)
service_summary.show(truncate=False)

+-------------+-----------------------+------------------------+-------------------------+------------------+------------------+
|service_level|total_shipping_fees_inr|total_shipping_costs_inr|shipments_with_known_cost|profit_inr        |profit_margin_pct |
+-------------+-----------------------+------------------------+-------------------------+------------------+------------------+
|Express      |1.0058468059999986E7   |7350805.699999983       |20689                    |2707662.360000002 |26.919232072403737|
|Standard     |1.3437372619999977E7   |1.0110465900000025E7    |39542                    |3326906.7199999522|24.758610288504137|
|Economy      |4489918.680000002      |3386212.500000007       |13266                    |1103706.1799999946|24.581874609809063|
+-------------+-----------------------+------------------------+-------------------------+------------------+------------------+



In [10]:
shipments_by_region = (
    shipments_df.join(warehouse_df.select("warehouse_id", "region"), "warehouse_id", "left")
    .groupBy("region")
    .agg(F.count("*").alias("shipment_count"))
    .orderBy(F.col("shipment_count").desc())
)
shipments_by_region.show(truncate=False)

+------+--------------+
|region|shipment_count|
+------+--------------+
|South |33018         |
|West  |26909         |
|North |9767          |
|East  |5306          |
+------+--------------+



In [11]:
carrier_cost_per_km = (
    shipments_df.join(carrier_df, "carrier_id", "left")
    .join(route_df.select("route_id", "distance_km"), "route_id", "left")
    .filter(
        F.col("shipping_cost_inr").isNotNull() &
        F.col("distance_km").isNotNull() &
        (F.col("distance_km") > 0)
    )
    .withColumn("cost_per_km", F.col("shipping_cost_inr") / F.col("distance_km"))
    .groupBy("carrier_id", "carrier_name")
    .agg(F.avg("cost_per_km").alias("avg_shipping_cost_per_km"))
    .orderBy("avg_shipping_cost_per_km")
)
carrier_cost_per_km.show(truncate=False)

+----------+--------------------+------------------------+
|carrier_id|carrier_name        |avg_shipping_cost_per_km|
+----------+--------------------+------------------------+
|C06       |RouteWise           |0.5879324060720517      |
|C05       |CargoNest           |0.5897283901696387      |
|C03       |MetroRoute Logistics|0.594275960971091       |
|C02       |ParcelBridge        |0.5957409571811839      |
|C01       |SwiftLine Express   |0.6032748764541479      |
|C07       |BluePeak Delivery   |0.6074832283099014      |
|C04       |AirSprint Couriers  |0.6125755103088726      |
|NULL      |NULL                |0.6318401277501529      |
+----------+--------------------+------------------------+



In [12]:
cols_to_check = ["carrier_id", "package_weight_kg", "shipping_cost_inr", "customer_rating"]
shipments_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c + "_missing")
    for c in cols_to_check
]).show()

+------------------+-------------------------+-------------------------+-----------------------+
|carrier_id_missing|package_weight_kg_missing|shipping_cost_inr_missing|customer_rating_missing|
+------------------+-------------------------+-------------------------+-----------------------+
|               871|                     1129|                     1503|                  26255|
+------------------+-------------------------+-------------------------+-----------------------+



In [13]:
handling_zero_df = shipments_df.fillna({"handling_minutes": 0})
known_carrier_df = shipments_df.filter(F.col("carrier_id").isNotNull())
print("Original rows:", shipments_df.count())
print("Rows after handling_minutes fill:", handling_zero_df.count())
print("Rows with known carrier_id:", known_carrier_df.count())

Original rows: 75000
Rows after handling_minutes fill: 75000
Rows with known carrier_id: 74129


In [14]:
weight_category_df = shipments_df.withColumn(
    "weight_category",
    F.when(F.col("package_weight_kg").isNull(), "Unknown")
     .when(F.col("package_weight_kg") < 2, "Light")
     .when(F.col("package_weight_kg") <= 10, "Medium")
     .otherwise("Heavy")
)
weight_category_df.select("shipment_id", "package_weight_kg", "weight_category").show(20, truncate=False)

+-----------+-----------------+---------------+
|shipment_id|package_weight_kg|weight_category|
+-----------+-----------------+---------------+
|SHP0000001 |3.93             |Medium         |
|SHP0000002 |5.62             |Medium         |
|SHP0000003 |8.78             |Medium         |
|SHP0000004 |0.97             |Light          |
|SHP0000005 |NULL             |Unknown        |
|SHP0000006 |6.05             |Medium         |
|SHP0000007 |1.36             |Light          |
|SHP0000008 |2.04             |Medium         |
|SHP0000009 |18.73            |Heavy          |
|SHP0000010 |2.02             |Medium         |
|SHP0000011 |6.87             |Medium         |
|SHP0000012 |11.83            |Heavy          |
|SHP0000013 |3.73             |Medium         |
|SHP0000014 |1.42             |Light          |
|SHP0000015 |1.74             |Light          |
|SHP0000016 |6.87             |Medium         |
|SHP0000017 |2.58             |Medium         |
|SHP0000018 |1.85             |Light    

In [15]:
date_cols = ["booking_date", "dispatch_date", "promised_delivery_date", "actual_delivery_date"]
typed_dates_df = shipments_df
for c in date_cols:
    typed_dates_df = typed_dates_df.withColumn(c, F.to_date(F.col(c)))
delivered_duration_df = (
    typed_dates_df.filter(
        (F.col("shipment_status") == "Delivered") &
        F.col("actual_delivery_date").isNotNull() &
        F.col("dispatch_date").isNotNull()
    )
    .withColumn("delivery_duration_days", F.datediff("actual_delivery_date", "dispatch_date"))
)
delivered_duration_df.select(
    "shipment_id", "dispatch_date", "actual_delivery_date", "delivery_duration_days"
).show(20, truncate=False)

+-----------+-------------+--------------------+----------------------+
|shipment_id|dispatch_date|actual_delivery_date|delivery_duration_days|
+-----------+-------------+--------------------+----------------------+
|SHP0000001 |2026-05-24   |2026-05-27          |3                     |
|SHP0000002 |2026-05-18   |2026-05-21          |3                     |
|SHP0000003 |2026-02-01   |2026-02-04          |3                     |
|SHP0000004 |2026-03-12   |2026-03-18          |6                     |
|SHP0000005 |2026-08-16   |2026-08-29          |13                    |
|SHP0000006 |2026-02-08   |2026-02-09          |1                     |
|SHP0000009 |2026-08-12   |2026-08-21          |9                     |
|SHP0000010 |2026-06-02   |2026-06-11          |9                     |
|SHP0000011 |2026-04-27   |2026-05-01          |4                     |
|SHP0000013 |2026-06-26   |2026-06-29          |3                     |
|SHP0000014 |2026-04-07   |2026-04-09          |2               

In [16]:
delivery_performance_df = typed_dates_df.withColumn(
    "delivery_performance",
    F.when(F.col("actual_delivery_date").isNull(), "Missing Actual Date")
     .when(F.col("actual_delivery_date") < F.col("promised_delivery_date"), "Early")
     .when(F.col("actual_delivery_date") == F.col("promised_delivery_date"), "On Time")
     .otherwise("Late")
)
delivery_performance_df.groupBy("delivery_performance").count().show()

+--------------------+-----+
|delivery_performance|count|
+--------------------+-----+
|             On Time|22049|
| Missing Actual Date|22566|
|                Late|21900|
|               Early| 8485|
+--------------------+-----+



In [17]:
monthly_delivered_df = (
    typed_dates_df.filter(F.col("shipment_status") == "Delivered")
    .withColumn("booking_month", F.date_format("booking_date", "yyyy-MM"))
    .groupBy("booking_month")
    .agg(
        F.count("*").alias("delivered_shipment_count"),
        F.sum("shipping_fee_inr").alias("total_shipping_fees_inr")
    )
    .orderBy("booking_month")
)
monthly_delivered_df.show(truncate=False)

+-------------+------------------------+-----------------------+
|booking_month|delivered_shipment_count|total_shipping_fees_inr|
+-------------+------------------------+-----------------------+
|2026-01      |6710                    |2525138.440000005      |
|2026-02      |6097                    |2268113.5000000037     |
|2026-03      |6810                    |2530456.2400000026     |
|2026-04      |6379                    |2389163.4600000014     |
|2026-05      |6858                    |2549683.9999999963     |
|2026-06      |6615                    |2447590.1200000006     |
|2026-07      |6773                    |2512355.2199999997     |
|2026-08      |6192                    |2306432.5199999996     |
+-------------+------------------------+-----------------------+



In [18]:
region_carrier_fees = (
    shipments_df.join(warehouse_df.select("warehouse_id", "region"), "warehouse_id", "left")
    .groupBy("region", "carrier_id")
    .agg(F.sum("shipping_fee_inr").alias("total_shipping_fees_inr"))
)
w_rank = Window.partitionBy("region").orderBy(F.col("total_shipping_fees_inr").desc())
ranked_carriers_df = region_carrier_fees.withColumn("carrier_rank", F.dense_rank().over(w_rank))
ranked_carriers_df.orderBy("region", "carrier_rank").show(truncate=False)

+------+----------+-----------------------+------------+
|region|carrier_id|total_shipping_fees_inr|carrier_rank|
+------+----------+-----------------------+------------+
|East  |C01       |393862.61999999994     |1           |
|East  |C02       |373590.02              |2           |
|East  |C03       |302260.92000000004     |3           |
|East  |C05       |243322.91999999998     |4           |
|East  |C07       |222025.85999999993     |5           |
|East  |C04       |211583.64000000004     |6           |
|East  |C06       |195810.3399999999      |7           |
|East  |NULL      |24594.6                |8           |
|North |C02       |731442.0999999994      |1           |
|North |C01       |653223.6200000005      |2           |
|North |C03       |597883.4600000005      |3           |
|North |C05       |464601.85999999975     |4           |
|North |C07       |425396.69999999995     |5           |
|North |C04       |393181.07999999984     |6           |
|North |C06       |338296.45999

In [19]:
w_latest = Window.partitionBy("warehouse_id").orderBy(
    F.col("booking_date").desc_nulls_last(),
    F.col("shipment_id").desc()
)
latest_shipment_df = (
    shipments_df.withColumn("row_num", F.row_number().over(w_latest))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)
latest_shipment_df.select("warehouse_id", "shipment_id", "booking_date").orderBy("warehouse_id").show(truncate=False)

+------------+-----------+------------+
|warehouse_id|shipment_id|booking_date|
+------------+-----------+------------+
|W01         |SHP0072346 |2026-08-28  |
|W02         |SHP0074049 |2026-08-28  |
|W03         |SHP0073860 |2026-08-28  |
|W04         |SHP0073268 |2026-08-28  |
|W05         |SHP0074305 |2026-08-28  |
|W06         |SHP0074983 |2026-08-28  |
|W07         |SHP0071003 |2026-08-28  |
|W08         |SHP0072690 |2026-08-28  |
+------------+-----------+------------+



In [20]:
w_running = (
    Window.partitionBy("warehouse_id")
    .orderBy(F.col("booking_date").asc_nulls_last(), F.col("shipment_id").asc())
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
running_fees_df = shipments_df.withColumn(
    "running_shipping_fees_inr", F.sum("shipping_fee_inr").over(w_running)
)
running_fees_df.select(
    "warehouse_id", "booking_date", "shipment_id", "shipping_fee_inr", "running_shipping_fees_inr"
).orderBy("warehouse_id", "booking_date", "shipment_id").show(30, truncate=False)

+------------+------------+-----------+----------------+-------------------------+
|warehouse_id|booking_date|shipment_id|shipping_fee_inr|running_shipping_fees_inr|
+------------+------------+-----------+----------------+-------------------------+
|W01         |2026-01-01  |SHP0004062 |312.48          |312.48                   |
|W01         |2026-01-01  |SHP0006739 |227.02          |539.5                    |
|W01         |2026-01-01  |SHP0007553 |316.02          |855.52                   |
|W01         |2026-01-01  |SHP0008505 |572.68          |1428.1999999999998       |
|W01         |2026-01-01  |SHP0009674 |322.9           |1751.1                   |
|W01         |2026-01-01  |SHP0011312 |390.1           |2141.2                   |
|W01         |2026-01-01  |SHP0011588 |421.72          |2562.92                  |
|W01         |2026-01-01  |SHP0011698 |190.22          |2753.14                  |
|W01         |2026-01-01  |SHP0011973 |384.64          |3137.7799999999997       |
|W01

In [21]:
combined_df = shipments_df.unionByName(extra_df)
print("Main count:", shipments_df.count())
print("Extra count:", extra_df.count())
print("Combined count:", combined_df.count())

Main count: 75000
Extra count: 12000
Combined count: 87000


In [22]:
repeated_ids_df = (
    combined_df.groupBy("shipment_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc(), "shipment_id")
)
repeated_ids_df.show(truncate=False)
before_dedup = combined_df.count()
deduplicated_df = combined_df.dropDuplicates(["shipment_id"])
after_dedup = deduplicated_df.count()
print("Rows before deduplication:", before_dedup)
print("Rows after deduplication:", after_dedup)
print("Rows removed:", before_dedup - after_dedup)

+-----------+-----+
|shipment_id|count|
+-----------+-----+
|SHP0000128 |2    |
|SHP0000254 |2    |
|SHP0000411 |2    |
|SHP0000455 |2    |
|SHP0000477 |2    |
|SHP0000550 |2    |
|SHP0000628 |2    |
|SHP0000774 |2    |
|SHP0000968 |2    |
|SHP0001041 |2    |
|SHP0001119 |2    |
|SHP0001166 |2    |
|SHP0001183 |2    |
|SHP0001222 |2    |
|SHP0001286 |2    |
|SHP0001448 |2    |
|SHP0001519 |2    |
|SHP0001535 |2    |
|SHP0001642 |2    |
|SHP0001737 |2    |
+-----------+-----+
only showing top 20 rows
Rows before deduplication: 87000
Rows after deduplication: 86250
Rows removed: 750


In [24]:
sensor_schema = StructType([
    StructField("warehouse_id", StringType(), True),
    StructField("environment", StructType([
        StructField("temperature_range_c", StructType([
            StructField("minimum", IntegerType(), True),
            StructField("maximum", IntegerType(), True)
        ]), True),
        StructField("humidity_target_pct", IntegerType(), True)
    ]), True),
    StructField("sensors", ArrayType(StructType([
        StructField("sensor_id", StringType(), True),
        StructField("type", StringType(), True),
        StructField("enabled", BooleanType(), True)
    ])), True)
])
sensor_df = spark.read.schema(sensor_schema).option("multiLine", True).json("warehouse_sensor_config.json")
sensor_details_df = (
    sensor_df
    .withColumn("sensor", F.explode("sensors"))
    .select(
        "warehouse_id",
        F.col("environment.temperature_range_c.minimum").alias("temperature_min_c"),
        F.col("environment.temperature_range_c.maximum").alias("temperature_max_c"),
        F.col("sensor.sensor_id").alias("sensor_id"),
        F.col("sensor.type").alias("sensor_type"),
        F.col("sensor.enabled").alias("sensor_enabled")
    )
)
sensor_details_df.show(truncate=False)

+------------+-----------------+-----------------+---------+-----------+--------------+
|warehouse_id|temperature_min_c|temperature_max_c|sensor_id|sensor_type|sensor_enabled|
+------------+-----------------+-----------------+---------+-----------+--------------+
|W01         |18               |32               |W01-S1   |Temperature|false         |
|W01         |18               |32               |W01-S2   |Humidity   |true          |
|W01         |18               |32               |W01-S3   |Door       |true          |
|W02         |18               |32               |W02-S1   |Temperature|true          |
|W02         |18               |32               |W02-S2   |Humidity   |true          |
|W02         |18               |32               |W02-S3   |Door       |true          |
|W03         |18               |32               |W03-S1   |Temperature|true          |
|W03         |18               |32               |W03-S2   |Humidity   |true          |
|W03         |18               |

In [25]:
pipeline_df = (
    shipments_df.filter(F.col("shipment_status") == "Delivered")
    .join(warehouse_df.select("warehouse_id", "region"), "warehouse_id", "left")
    .withColumn("estimated_profit_inr", F.col("shipping_fee_inr") - F.col("shipping_cost_inr"))
    .groupBy("region", "service_level")
    .agg(
        F.count("*").alias("delivered_shipments"),
        F.sum("shipping_fee_inr").alias("total_fees_inr"),
        F.sum("estimated_profit_inr").alias("estimated_profit_inr")
    )
    .orderBy("region", "service_level")
)
pipeline_df.explain(True)  # Inspect logical and physical plans before action.
pipeline_df.show(truncate=False)  # Action triggers execution.

== Parsed Logical Plan ==
'Sort ['region ASC NULLS FIRST, 'service_level ASC NULLS FIRST], true
+- Aggregate [region#94, service_level#26], [region#94, service_level#26, count(1) AS delivered_shipments#1256L, sum(shipping_fee_inr#28) AS total_fees_inr#1257, sum(estimated_profit_inr#1255) AS estimated_profit_inr#1258]
   +- Project [warehouse_id#22, shipment_id#17, booking_date#18, dispatch_date#19, promised_delivery_date#20, actual_delivery_date#21, carrier_id#23, route_id#24, customer_segment#25, service_level#26, package_weight_kg#27, shipping_fee_inr#28, shipping_cost_inr#29, shipment_status#30, delivery_attempts#31, damage_flag#32, customer_rating#33, handling_minutes#34, payment_mode#35, priority_flag#36, region#94, (shipping_fee_inr#28 - shipping_cost_inr#29) AS estimated_profit_inr#1255]
      +- Project [warehouse_id#22, shipment_id#17, booking_date#18, dispatch_date#19, promised_delivery_date#20, actual_delivery_date#21, carrier_id#23, route_id#24, customer_segment#25, service

In [26]:
delivered_df = shipments_df.filter(F.col("shipment_status") == "Delivered").cache()
start = time.perf_counter()
count_first = delivered_df.count()
time_count_first = time.perf_counter() - start
start = time.perf_counter()
count_second = delivered_df.count()
time_count_second = time.perf_counter() - start
start = time.perf_counter()
delivered_df.groupBy("service_level").agg(F.sum("shipping_fee_inr").alias("fees")).show()
time_agg = time.perf_counter() - start
print(f"First count: {count_first}, elapsed: {time_count_first:.4f}s")
print(f"Second count: {count_second}, elapsed: {time_count_second:.4f}s")
print(f"Aggregation elapsed: {time_agg:.4f}s")
delivered_df.unpersist()

+-------------+------------------+
|service_level|              fees|
+-------------+------------------+
|      Express|7053103.1200000085|
|     Standard| 9350271.759999998|
|      Economy|3125558.6200000015|
+-------------+------------------+

First count: 52434, elapsed: 2.0393s
Second count: 52434, elapsed: 0.0980s
Aggregation elapsed: 0.3163s


DataFrame[shipment_id: string, booking_date: date, dispatch_date: date, promised_delivery_date: date, actual_delivery_date: date, warehouse_id: string, carrier_id: string, route_id: string, customer_segment: string, service_level: string, package_weight_kg: double, shipping_fee_inr: double, shipping_cost_inr: double, shipment_status: string, delivery_attempts: int, damage_flag: string, customer_rating: double, handling_minutes: double, payment_mode: string, priority_flag: string]

In [27]:
express_df = shipments_df.filter(F.col("service_level") == "Express").persist(StorageLevel.MEMORY_AND_DISK)
print("Express count:", express_df.count())  # Action 1
express_df.groupBy("shipment_status").count().show()  # Action 2
print("Storage level:", express_df.storageLevel)
express_df.unpersist()

Express count: 21090
+---------------+-----+
|shipment_status|count|
+---------------+-----+
|       Returned| 1767|
|           Lost|  575|
|     In Transit| 2944|
|      Cancelled|  985|
|      Delivered|14819|
+---------------+-----+

Storage level: Disk Memory Serialized 1x Replicated


DataFrame[shipment_id: string, booking_date: date, dispatch_date: date, promised_delivery_date: date, actual_delivery_date: date, warehouse_id: string, carrier_id: string, route_id: string, customer_segment: string, service_level: string, package_weight_kg: double, shipping_fee_inr: double, shipping_cost_inr: double, shipment_status: string, delivery_attempts: int, damage_flag: string, customer_rating: double, handling_minutes: double, payment_mode: string, priority_flag: string]

In [28]:
from pyspark.sql.functions import spark_partition_id
shipments_8_df = shipments_df.repartition(8)
partition_counts_8 = (
    shipments_8_df.withColumn("partition_id", spark_partition_id())
    .groupBy("partition_id").count()
    .orderBy("partition_id")
)
partition_counts_8.show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 9375|
|           1| 9374|
|           2| 9376|
|           3| 9375|
|           4| 9375|
|           5| 9375|
|           6| 9375|
|           7| 9375|
+------------+-----+



In [29]:
shipments_12_df = shipments_df.repartition(12)
coalesced_3_df = shipments_12_df.coalesce(3)
repartitioned_3_df = shipments_12_df.repartition(3)
print("Plan for coalesce(3):")
coalesced_3_df.explain(True)
print("Plan for repartition(3):")
repartitioned_3_df.explain(True)
print("Partition count after coalesce:", coalesced_3_df.rdd.getNumPartitions())
print("Partition count after repartition:", repartitioned_3_df.rdd.getNumPartitions())

Plan for coalesce(3):
== Parsed Logical Plan ==
Repartition 3, false
+- Repartition 12, true
   +- Relation [shipment_id#17,booking_date#18,dispatch_date#19,promised_delivery_date#20,actual_delivery_date#21,warehouse_id#22,carrier_id#23,route_id#24,customer_segment#25,service_level#26,package_weight_kg#27,shipping_fee_inr#28,shipping_cost_inr#29,shipment_status#30,delivery_attempts#31,damage_flag#32,customer_rating#33,handling_minutes#34,payment_mode#35,priority_flag#36] csv

== Analyzed Logical Plan ==
shipment_id: string, booking_date: date, dispatch_date: date, promised_delivery_date: date, actual_delivery_date: date, warehouse_id: string, carrier_id: string, route_id: string, customer_segment: string, service_level: string, package_weight_kg: double, shipping_fee_inr: double, shipping_cost_inr: double, shipment_status: string, delivery_attempts: int, damage_flag: string, customer_rating: double, handling_minutes: double, payment_mode: string, priority_flag: string
Repartition 3, fa

In [30]:
shipment_warehouse_df = shipments_df.join(
    warehouse_df, on="warehouse_id", how="left"
).repartition(4)
output_path = "/content/shipment_warehouse_parquet"
shipment_warehouse_df.write.mode("overwrite").parquet(output_path)
print("Saved Parquet to:", output_path)
print("Generated files:")
for root, dirs, filenames in os.walk(output_path):
    for filename in filenames:
        print(os.path.join(root, filename))

Saved Parquet to: /content/shipment_warehouse_parquet
Generated files:
/content/shipment_warehouse_parquet/.part-00002-3f4a2211-38cf-418b-92b3-421fb02f9b7d-c000.snappy.parquet.crc
/content/shipment_warehouse_parquet/part-00001-3f4a2211-38cf-418b-92b3-421fb02f9b7d-c000.snappy.parquet
/content/shipment_warehouse_parquet/._SUCCESS.crc
/content/shipment_warehouse_parquet/.part-00001-3f4a2211-38cf-418b-92b3-421fb02f9b7d-c000.snappy.parquet.crc
/content/shipment_warehouse_parquet/.part-00000-3f4a2211-38cf-418b-92b3-421fb02f9b7d-c000.snappy.parquet.crc
/content/shipment_warehouse_parquet/_SUCCESS
/content/shipment_warehouse_parquet/part-00000-3f4a2211-38cf-418b-92b3-421fb02f9b7d-c000.snappy.parquet
/content/shipment_warehouse_parquet/part-00003-3f4a2211-38cf-418b-92b3-421fb02f9b7d-c000.snappy.parquet
/content/shipment_warehouse_parquet/part-00002-3f4a2211-38cf-418b-92b3-421fb02f9b7d-c000.snappy.parquet
/content/shipment_warehouse_parquet/.part-00003-3f4a2211-38cf-418b-92b3-421fb02f9b7d-c000.s

In [31]:
all_shipments = shipments_df.unionByName(extra_df).dropDuplicates(["shipment_id"])

In [32]:
for c in ["booking_date", "dispatch_date", "promised_delivery_date", "actual_delivery_date"]:
    all_shipments = all_shipments.withColumn(c, F.to_date(F.col(c)))

In [33]:
cost_median = all_shipments.approxQuantile("shipping_cost_inr", [0.5], 0.01)
weight_median = all_shipments.approxQuantile("package_weight_kg", [0.5], 0.01)
if not cost_median:
    raise ValueError("shipping_cost_inr has no non-null values; define a fallback imputation rule.")
if not weight_median:
    raise ValueError("package_weight_kg has no non-null values; define a fallback imputation rule.")
cleaned_shipments = (
    all_shipments
    .withColumn("shipping_cost_clean", F.coalesce(F.col("shipping_cost_inr"), F.lit(float(cost_median[0]))))
    .withColumn("package_weight_clean", F.coalesce(F.col("package_weight_kg"), F.lit(float(weight_median[0]))))
    .withColumn(
        "delivery_duration_days",
        F.when(
            F.col("actual_delivery_date").isNotNull() & F.col("dispatch_date").isNotNull(),
            F.datediff("actual_delivery_date", "dispatch_date")
        )
    )
    .withColumn(
        "late_delivery",
        F.when(
            F.col("actual_delivery_date").isNull() | F.col("promised_delivery_date").isNull(),
            F.lit(None).cast("integer")
        ).when(F.col("actual_delivery_date") > F.col("promised_delivery_date"), 1).otherwise(0)
    )
    .withColumn("shipment_profit_inr", F.col("shipping_fee_inr") - F.col("shipping_cost_clean"))
)

In [34]:
performance_detail = (
    cleaned_shipments
    .join(warehouse_df.select("warehouse_id", "region"), "warehouse_id", "left")
    .join(carrier_df.select("carrier_id", "carrier_name", "reliability_score"), "carrier_id", "left")
    .join(route_df.select("route_id", "distance_km", "toll_inr", "route_type"), "route_id", "left")
    .withColumn(
        "on_time_or_early",
        F.when(
            F.col("actual_delivery_date").isNull() | F.col("promised_delivery_date").isNull(),
            F.lit(None).cast("integer")
        ).when(F.col("actual_delivery_date") <= F.col("promised_delivery_date"), 1).otherwise(0)
    )
)

In [35]:
final_report = (
    performance_detail
    .groupBy("region", "carrier_id", "carrier_name")
    .agg(
        F.count("*").alias("shipment_count"),
        F.sum("shipping_fee_inr").alias("revenue_inr"),
        F.sum("shipment_profit_inr").alias("profit_inr"),
        (F.avg("late_delivery") * 100).alias("late_delivery_pct"),
        (F.avg("on_time_or_early") * 100).alias("on_time_delivery_pct"),
        F.sum(F.when(F.col("actual_delivery_date").isNull(), 1).otherwise(0)).alias("missing_actual_date_count")
    )
)

In [36]:
w_on_time_rank = Window.partitionBy("region").orderBy(
    F.col("on_time_delivery_pct").desc_nulls_last()
)
final_report = final_report.withColumn("on_time_rank_in_region", F.dense_rank().over(w_on_time_rank))

In [37]:
final_report = final_report.cache()
print("Final report rows:", final_report.count())  # Action 1
final_report.orderBy("region", "on_time_rank_in_region").show(100, truncate=False)  # Action 2
final_report.groupBy("region").agg(
    F.sum("revenue_inr").alias("region_revenue_inr"),
    F.sum("profit_inr").alias("region_profit_inr")
).show(truncate=False)

Final report rows: 32
+------+----------+--------------------+--------------+------------------+------------------+------------------+--------------------+-------------------------+----------------------+
|region|carrier_id|carrier_name        |shipment_count|revenue_inr       |profit_inr        |late_delivery_pct |on_time_delivery_pct|missing_actual_date_count|on_time_rank_in_region|
+------+----------+--------------------+--------------+------------------+------------------+------------------+--------------------+-------------------------+----------------------+
|East  |C01       |SwiftLine Express   |1190          |452813.2599999999 |110235.76000000002|38.47072879330944 |61.52927120669056   |353                      |1                     |
|East  |C04       |AirSprint Couriers  |648           |243239.28000000012|59472.08          |40.0              |60.0                |213                      |2                     |
|East  |C05       |CargoNest           |765           |283412.3

In [38]:
final_output_path = "/content/logistics_performance_report"
final_report.write.mode("overwrite").partitionBy("region").parquet(final_output_path)
print("Partitioned Parquet report saved to:", final_output_path)

Partitioned Parquet report saved to: /content/logistics_performance_report


In [39]:
for root, dirs, filenames in os.walk(final_output_path):
    for filename in filenames:
        print(os.path.join(root, filename))
final_report.unpersist()

/content/logistics_performance_report/._SUCCESS.crc
/content/logistics_performance_report/_SUCCESS
/content/logistics_performance_report/region=West/part-00164-256c39c8-f1e7-4c18-8786-50b594290bca.c000.snappy.parquet
/content/logistics_performance_report/region=West/.part-00164-256c39c8-f1e7-4c18-8786-50b594290bca.c000.snappy.parquet.crc
/content/logistics_performance_report/region=North/part-00192-256c39c8-f1e7-4c18-8786-50b594290bca.c000.snappy.parquet
/content/logistics_performance_report/region=North/.part-00192-256c39c8-f1e7-4c18-8786-50b594290bca.c000.snappy.parquet.crc
/content/logistics_performance_report/region=South/part-00090-256c39c8-f1e7-4c18-8786-50b594290bca.c000.snappy.parquet
/content/logistics_performance_report/region=South/.part-00090-256c39c8-f1e7-4c18-8786-50b594290bca.c000.snappy.parquet.crc
/content/logistics_performance_report/region=East/.part-00129-256c39c8-f1e7-4c18-8786-50b594290bca.c000.snappy.parquet.crc
/content/logistics_performance_report/region=East/p

DataFrame[region: string, carrier_id: string, carrier_name: string, shipment_count: bigint, revenue_inr: double, profit_inr: double, late_delivery_pct: double, on_time_delivery_pct: double, missing_actual_date_count: bigint, on_time_rank_in_region: int]